In [0]:
%pip install requests

In [0]:
import requests
import json


API_KEY = "e0d9d069-c5e9-4397-97c6-6f3f7d79220e"

company_number = "00002065"

base_url = f"https://api.company-information.service.gov.uk/company/{company_number}"

response = requests.get(base_url, auth=(API_KEY, ""))

data = response.json()

In [0]:
import requests
import time
import json
import os

API_KEY = "e0d9d069-c5e9-4397-97c6-6f3f7d79220e"

company_numbers = [
    "04366849", # Shell plc
    "00617987", # HSBC Holdings plc
    "00102498", # BP p.l.c.
    "00041424", # Unilever PLC
    "01833679", # Vodafone Group Plc
    "00048839", # Barclays PLC
    "02723534", # AstraZeneca PLC
    "02231246", # The Sage Group plc
    "00502851", # Greggs plc
    "01372603", # Bellway plc
    "01106260", # Renishaw plc
    "02366640", # Pennon Group plc
    "02366619", # Severn Trent plc
    "00714275", # IMI plc
    "00395826", # Balfour Beatty plc
    "05562053", # Drax Group plc
    "03263464", # Telecom Plus plc
    "03033654", # Centrica plc
    "04627044", # NCC Group plc
    "00185647" # J Sainsbury plc
]

from pyspark.sql.functions import current_date, year, month, dayofmonth

df_today = spark.range(1).select(current_date().alias("today"))


results = df_today.select(year("today").alias("year"),
          month("today").alias("month"),
          dayofmonth("today").alias("day")).collect()[0]

year = results["year"]
month = results["month"]
day = results["day"]

# Base output directory (change this as needed)
ROOT_PATH = f"/Volumes/corporate_data_lakehouse/bronze/raw_comp_data/company_house"

# Create folder structure
folders = {
    "overview": os.path.join(ROOT_PATH, "overview", str(year), str(month), str(day)),
    "filing_history": os.path.join(ROOT_PATH, "filing_history", str(year), str(month), str(day)),
    "people": os.path.join(ROOT_PATH, "people", str(year), str(month), str(day))
}

for folder in folders.values():
    os.makedirs(folder, exist_ok=True)


def fetch_and_save(url, file_path):
    try:
        response = requests.get(url, auth=(API_KEY, ""))
        
        if response.status_code == 200:
            with open(file_path, "w", encoding="utf-8") as f:
                json.dump(response.json(), f, indent=2)
        else:
            print(f"Failed ({response.status_code}): {url}")
    
    except Exception as e:
        print(f"Error calling API: {e}")


for c in company_numbers:
    print(f"Processing company: {c}")

    # -------- Overview --------
    overview_url = f"https://api.company-information.service.gov.uk/company/{c}"
    fetch_and_save(
        overview_url,
        os.path.join(folders["overview"], f"{c}.json")
    )

    # -------- Filing History --------
    filing_url = f"https://api.company-information.service.gov.uk/company/{c}/filing-history"
    fetch_and_save(
        filing_url,
        os.path.join(folders["filing_history"], f"{c}.json")
    )

    # --------  Officers --------
    people_url = f"https://api.company-information.service.gov.uk/company/{c}/officers"
    fetch_and_save(
        people_url,
        os.path.join(folders["people"], f"{c}.json")
    )
    #count oficers and more logging

    # Rate limiting (Companies House recommends this)
    time.sleep(0.5)

print("Data ingestion complete")